# fin-news-sentiment-lab: GPU runner (Colab / Kaggle)

<a href="https://colab.research.google.com/github/samarthkala5/fin-news-sentiment-lab/blob/main/notebooks/colab_runner.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A **thin wrapper**: every code cell runs a script from `scripts/`. All the logic lives in
`scripts/` and `src/fnsl/`, where it is tested. Each training script writes
`results/metrics/<model>.json`, `results/predictions/<model>.csv` and
`results/figures/<model>_confusion.png`; step 9 zips them so you can copy them into your local
repo and commit them there.

**Before you start**
- **Colab:** Runtime > Change runtime type > **T4 GPU**.
- **Kaggle:** Settings > Accelerator > **GPU T4 x2** (not P100, which Unsloth does not
  support) and Settings > Internet > **on**.
- Run steps 1-3 at the start of every session. Steps 4-8 are independent: run only what you
  need today.
- Nothing private is mounted and no token is needed: the repo and all models are public. A
  warning that `HF_TOKEN` is not set can be ignored.
- Colab deletes every file when the session ends, so download the zip (step 9) before you
  close the tab.

## 1. Check the GPU

`nvidia-smi` shows the GPU and its memory. The second line asks PyTorch whether it can use
CUDA. It runs in a separate Python process (`!python -c ...`), so this notebook never imports
torch itself, and the install in step 3 cannot leave an old torch loaded here.

In [ ]:
!nvidia-smi
!python -c "import torch; print('torch', torch.__version__, 'CUDA:', torch.cuda.is_available())"

## 2. Clone the repository

`BRANCH` picks what to run, for example a feature branch that is not merged yet;
`--depth 1` downloads only the latest commit.

The clone works without credentials only if the repository is **public**. If it is private,
make it public, or keep a GitHub token in a Colab secret (key icon in the left sidebar) and
read it from there. **Never paste a token into the notebook**: notebooks get saved, shared and
committed.

In [ ]:
import os

REPO_URL = "https://github.com/samarthkala5/fin-news-sentiment-lab.git"
BRANCH = "main"

if os.path.exists("scripts/train_distilbert.py"):
    print("Already inside the repo, nothing to clone")
elif os.path.isdir("fin-news-sentiment-lab"):
    print("Already cloned: fin-news-sentiment-lab/")
else:
    !git clone --depth 1 -b $BRANCH $REPO_URL

## 3. Install the GPU requirements

`requirements-gpu.txt` adds Unsloth, transformers, sentence-transformers, peft, trl, accelerate
and bitsandbytes to the light requirements (torch and TensorFlow come preinstalled on Colab and
Kaggle).
`pip install -e . --no-deps` makes `import fnsl` work. Expect this to take several minutes.

**Is a runtime restart needed?** Normally not. Every later step runs a script in a new process
(`!python ...`), which always sees the freshly installed packages, and this notebook never
imports them itself. If Colab still shows a *Restart session* button, restarting is harmless:
afterwards run steps 2 and 3 again (they skip what is already done), then continue.

In [ ]:
if not os.path.exists("scripts/train_distilbert.py"):
    %cd fin-news-sentiment-lab
!pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

Installed versions, read from the package metadata on disk (so they are right even without a
restart):

In [ ]:
from importlib.metadata import PackageNotFoundError, version

packages = ["torch", "transformers", "peft", "trl", "accelerate", "bitsandbytes", "unsloth"]
packages += ["datasets", "tensorflow", "keras"]
for package in packages:
    try:
        print(f"{package:>13} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:>13} not installed")

**If the install fails**, or replaces the preinstalled CUDA build of torch, try the
`--no-deps` install that Unsloth's own Colab notebooks use. It is adapted from those notebooks
and **has not been tested** with this repo:

```
!pip install -q sentencepiece protobuf "datasets>=4.7,<5" "huggingface_hub>=1.5,<2" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft "trl==1.15.0" triton unsloth
!pip install -q --no-deps "transformers==5.17.0" "tokenizers>=0.23.1,<0.24"
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
```

## 4. Optional: smoke runs

Tiny subsets of the splits and 1 epoch (or a few steps), to check that the code runs on this
machine before a long run. Smoke runs write only to `results/smoke/` and `checkpoints/smoke/`,
are never used as results, and are left out of the zip in step 9.

In [ ]:
!python scripts/train_distilbert.py --smoke
!python scripts/train_keras.py --smoke
!python scripts/train_unsloth.py --smoke --mode all

## 5. DistilBERT (Days 5-7)

Full fine-tune with the script's defaults: 3 epochs, batch 32, learning rate 2e-5 with linear
warm-up and decay, `max_length` 64, float16 mixed precision on the GPU. After every epoch the
model is scored on the **validation** split; the epoch with the best validation macro-F1 is
saved to `checkpoints/distilbert/best`, and only that checkpoint predicts the test split, once.
Check that the printed training loss falls from epoch to epoch.

Writes `results/metrics/distilbert.json`, `results/predictions/distilbert.csv` and
`results/figures/distilbert_confusion.png`.

In [ ]:
!python scripts/train_distilbert.py

**Optional (Day 7): two more seeds.** Shows how much the test score moves when only the random
seed changes (classifier-head initialisation, data order, dropout). The runs are saved as
`distilbert_seed43` and `distilbert_seed44` and are compared like any other model.

In [ ]:
!python scripts/train_distilbert.py --seed 43
!python scripts/train_distilbert.py --seed 44

## 6. Optional: Keras BiLSTM (Day 4)

The BiLSTM trains fine on a laptop CPU (`python scripts/train_keras.py`), so this cell is only
a fallback in case TensorFlow does not work locally. It uses the TensorFlow that comes with
Colab / Kaggle; if step 3 lists `tensorflow` as not installed, run it locally instead. Writes
the `keras_bilstm` results.

In [ ]:
!python scripts/train_keras.py

## 7. Small LLM with Unsloth (Days 8-10)

The default model is `unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit`, an instruct model
stored in 4 bits. Zero-shot, training and evaluation use the same prompt template, and
`--mode eval` refuses an adapter trained with a different prompt version. Answers that map to
no label count as wrong; their number and some examples go into the metrics JSON. On Kaggle's
GPU T4 x2 the script uses only the first GPU.

The LoRA adapter is saved to `checkpoints/llm_lora_adapter/` (the default `--adapter-dir`).
Colab deletes it when the session ends, so run Days 9 and 10 in the same session (together
they should take about 30 minutes on a T4: a rough estimate, not a measurement), or add the
adapter to the zip in step 9. To keep it somewhere else, pass the same `--adapter-dir <folder>`
to both the train and the eval command.

### Day 8: zero-shot baseline on 50 test headlines

In [ ]:
!python scripts/train_unsloth.py --mode zero-shot

Writes the `llm_zeroshot` results. They cover only 50 of the test rows, so the results table
marks them *not comparable* and McNemar cannot pair them with the other models (see the
optional full-test run under Day 10).

### Day 9: LoRA fine-tune

Script defaults: rank r=16, alpha 16, dropout 0, adapters on all attention and MLP projections
(q, k, v, o, gate, up, down), 300 steps, learning rate 2e-4, batch 8 x gradient accumulation 2.
The validation loss is measured on 200 validation rows before training and every 50 steps; the
test split is not touched. Writes the adapter and `results/metrics/llm_lora_train.json` (loss
curves, no test metrics).

In [ ]:
!python scripts/train_unsloth.py --mode train

### Day 10: evaluate the adapter on the whole test split

Writes the `llm_lora` results (metrics, predictions, confusion matrix).

In [ ]:
!python scripts/train_unsloth.py --mode eval

**Optional:** zero-shot on all test rows, so that McNemar can pair zero-shot with every other
model. This overwrites the 50-row Day 8 `llm_zeroshot` files.

In [ ]:
!python scripts/train_unsloth.py --mode zero-shot --n-test 0

## 8. Optional: compare models and print the tables

You can also do this locally after copying the results back, which is usually simpler: there
the comparison also sees the results you produced on your laptop. `compare_models.py` computes
bootstrap CIs, McNemar's test and Holm-adjusted p-values for every model with test predictions
in this copy of the repo and writes `results/metrics/comparison.json`. Without flags,
`make_results_table.py` only prints the tables.

In [ ]:
!python scripts/compare_models.py
!python scripts/make_results_table.py

## 9. Download the results

Zips `results/metrics`, `results/predictions` and `results/figures` (small JSON, CSV and PNG
files; `results/smoke/` is left out). Optionally add the LoRA adapter (tens of MB) and the best
DistilBERT checkpoint (about 270 MB, needed for `scripts/push_to_hub.py --push` on Day 14):
large files make the download slow. On Colab the browser downloads `results.zip`; on Kaggle it
is saved to `/kaggle/working/` (download it from the Output panel).

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

INCLUDE_LORA_ADAPTER = False  # checkpoints/llm_lora_adapter, tens of MB
INCLUDE_DISTILBERT_CHECKPOINT = False  # checkpoints/distilbert/best, about 270 MB

folders = ["results/metrics", "results/predictions", "results/figures"]
if INCLUDE_LORA_ADAPTER:
    folders.append("checkpoints/llm_lora_adapter")
if INCLUDE_DISTILBERT_CHECKPOINT:
    folders.append("checkpoints/distilbert/best")

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file():
                zf.write(path)  # stored with its path relative to the repo root
print(f"results.zip: {os.path.getsize('results.zip') / 1e6:.1f} MB")

try:
    from google.colab import files  # only exists on Colab

    files.download("results.zip")
except ImportError:  # Kaggle, or a local Jupyter
    if os.path.isdir("/kaggle/working"):
        shutil.copy("results.zip", "/kaggle/working/results.zip")
        print("Saved /kaggle/working/results.zip: download it from the Output panel")
    else:
        print("Saved results.zip in", os.getcwd())

## 10. Back on your laptop

```bash
cd fin-news-sentiment-lab
unzip -o ~/Downloads/results.zip    # files land in results/ (and checkpoints/ if included)
python scripts/compare_models.py
python scripts/make_results_table.py --update-readme --count-tests
pytest -m "not heavy" && ruff check .
git status                          # expect changes in results/, README.md, RESUME_NUMBERS.md
```

Then commit with a `<verb> <what>` message, for example
`evaluate distilbert, save predictions, reading log`. MLflow runs logged on Colab stay on
Colab; every number that matters is in `results/metrics/*.json`.